In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
(PROJECT_ROOT / "reports").mkdir(exist_ok=True)


# Preprocessing and age labels for building-age classification

Run this notebook from top to bottom after `eda.ipynb`. It reads the original Madrid and Amsterdam Parquet files and creates **auditable age labels, unscaled location features, spatial fold assignments, and few-shot adaptation samples**. It also demonstrates training-only imputation/scaling. It does not train a classifier or report F1.

**Why these choices matter:** repeated years of a pixel are not independent labeled examples; combining different scene dates band-by-band creates artificial spectra; globally fitted scalers leak validation information; and backfilling missing historical observations can manufacture evidence of construction change. This notebook prevents those problems through coherent observation selection, location-level aggregation, explicit missingness, and training-fold pipelines.

**Output contract:** `data/preprocessed_age_labels/` is a separate output directory. Existing numbered notebooks and `data/preprocessed/preprocessed_data.pkl` are not overwritten. The older modelling notebook must be adapted to this new schema; do not feed it arrays from different preprocessing versions.


## 1. Configuration and reproducibility

Configuration is explicit so organizers can change boundaries or observation years without editing hidden logic. Historical event years come from existing notebooks 2/3 and still require organizer confirmation. We reproduce their **right-closed** label bins. Class assignment is a deterministic rule, not a learned transformation.

The real files include 2025 observations despite the introduction describing 1984–2024. Default features use 1984–2024, with excluded rows counted. This is a retrospective classification task at that cutoff: older observations can predate the current building, so we do not filter time series using the target construction date. A forecasting task would need a different, causal feature design.


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import joblib
from IPython.display import display
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.utils.class_weight import compute_sample_weight

SEED = 42
BANDS = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']
KEY = ['city', 'px_key', 'py_key']
CLASS_EVENTS = {'Madrid': 1960, 'Amsterdam': 1945}
MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR = 1850, 2024
MIN_OBSERVATION_YEAR, MAX_OBSERVATION_YEAR = 1984, 2024
EARLY_END = 2003
MIN_COVERAGE = .15
N_FOLDS = 5
# Coordinate-key units, not a confirmed distance in metres.
BLOCK_SIZE_KEYS = 20
BUDGETS = [5, 25, 50, 100, 200]
# Indices need confirmation of reflectance scale AND offset. Raw bands remain usable.
ENABLE_INDICES = False
REFLECTANCE_ENCODING_CONFIRMED = False
REFLECTANCE_SCALE = 1.0
REFLECTANCE_OFFSET = 0.0
EXPORT = True
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'pyproject.toml').is_file() and (p/'data').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Run inside the project or configure ROOT.')
PATHS = {c: ROOT/'data'/f for c,
         f in [('Madrid', 'madrid_train.parquet'), ('Amsterdam', 'amsterdam_data.parquet')]}
OUTPUT = ROOT/'data'/'preprocessed_age_labels'
if EXPORT:
    OUTPUT.mkdir(parents=True, exist_ok=True)
assert MIN_OBSERVATION_YEAR <= EARLY_END < MAX_OBSERVATION_YEAR
assert N_FOLDS >= 2 and BLOCK_SIZE_KEYS > 0
if ENABLE_INDICES and not REFLECTANCE_ENCODING_CONFIRMED:
    raise ValueError(
        'Confirm upstream scale/offset before enabling normalized spectral indices.')
VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__,
            'pandas': pd.__version__, 'scikit_learn': sklearn.__version__}
print(VERSIONS)
print('Output directory:', OUTPUT)

## 2. Create age labels with explicit validity and boundary checks

`weighted_mean_year` is an area-weighted **mean**, not a majority-age class. Its discretization follows the provided workflow and remains a limitation for mixed-age pixels. Validity requires a finite year within 1850–2024 and `usable_label=True`. A location with inconsistent target years or any unusable target row is excluded from supervised training, rather than silently choosing one target.

Exact bins: class 1 ≤ city event; class 2 > event and ≤1984; class 3 >1984 and ≤2004; class 4 >2004 and ≤2024. Null, infinite and out-of-domain targets stay missing; they must never be filled by an imputer. Boundary checks below make endpoint behavior visible.


In [ ]:
def assign_age_class(years, city):
    values = pd.to_numeric(years, errors='coerce')
    valid = np.isfinite(values) & values.between(
        MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR)
    return pd.cut(values.where(valid),
                  [-np.inf, CLASS_EVENTS[city], 1984, 2004, MAX_CONSTRUCTION_YEAR],
                  labels=[1, 2, 3, 4], right=True).astype('Int64')


for city, event in CLASS_EVENTS.items():
    years = pd.Series([1849, 1850, event, event+.5, 1984,
                      1984.5, 2004, 2004.5, 2024, 2025, np.nan, np.inf])
    labels = assign_age_class(years, city)
    assert labels.iloc[[0, 9, 10, 11]].isna().all()
    assert labels.iloc[1:9].tolist() == [1, 1, 2, 2, 3, 3, 4, 4]
    display(pd.DataFrame(
        {'city': city, 'construction_year': years, 'age_class': labels}))

## 3. Load one city at a time and select coherent spectra

Read only required columns to reduce memory use. Fail on missing identifiers, unexpected cities, noninteger observation years, or duplicate location–year records: these require upstream review. Coverage filtering uses the challenge's fixed rule, not a threshold tuned on validation results.

For each pixel–year, choose the first slot where the QA flag is true and **all six bands are finite**. All bands come from that same slot. Preserve a separate audit trail for no-valid-observation rows. Do not clip spectral extremes using unverified physical units or discard rare classes by arbitrary quantiles.

Create row-level labels and a location-level label table independently of the spectral filter. Location IDs and labels are kept as metadata; the feature engineering function below receives spectra only, so it cannot use construction years as predictors.


In [ ]:
def select_coherent_observation(df):
    values = np.full((len(df), len(BANDS)), np.nan, dtype=np.float32)
    selected = np.zeros(len(df), dtype=np.int8)
    valid_slots = np.zeros(len(df), dtype=np.int8)
    incomplete_qa = 0
    for slot in [1, 2, 3]:
        array = df[[f'{b}_{slot}' for b in BANDS]].to_numpy(dtype=np.float32)
        qa = df[f'qa_valid_{slot}'].fillna(False).to_numpy(dtype=bool)
        complete = np.isfinite(array).all(axis=1)
        valid = qa & complete
        incomplete_qa += int((qa & ~complete).sum())
        valid_slots += valid
        pick = (selected == 0) & valid
        values[pick] = array[pick]
        selected[pick] = slot
    return values, selected, valid_slots, incomplete_qa


def prepare_city(city, path):
    metadata = KEY+['year', 'coverage', 'weighted_mean_year', 'usable_label']
    observation = [f'{b}_{s}' for s in [1, 2, 3]
                   for b in BANDS]+[f'qa_valid_{s}' for s in [1, 2, 3]]
    df = pd.read_parquet(path, columns=metadata+observation)
    if df[KEY+['year']].isna().any().any():
        raise ValueError(f'{city}: null location/year key')
    if not df.city.eq(city).all():
        raise ValueError(f'{city}: unexpected city names')
    if not np.isfinite(df.year).all() or not df.year.mod(1).eq(0).all():
        raise ValueError(f'{city}: acquisition years must be finite integers')
    if df.duplicated(KEY+['year']).any():
        raise ValueError(f'{city}: duplicate location-year records')
    year_ok = df.weighted_mean_year.between(
        MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR)
    df['row_label_valid'] = year_ok & df.usable_label.fillna(False)
    label_groups = df.groupby(KEY, sort=True, observed=True)
    loc = label_groups.agg(weighted_mean_year=('weighted_mean_year', 'first'),
                           target_distinct_including_null=(
                               'weighted_mean_year', lambda s: s.nunique(dropna=False)),
                           all_rows_label_valid=('row_label_valid', 'all')).reset_index()
    loc['label_eligible'] = loc.all_rows_label_valid & loc.target_distinct_including_null.eq(
        1)
    loc['age_class'] = assign_age_class(
        loc.weighted_mean_year, city).where(loc.label_eligible)
    # A stable, readable identifier makes cross-file alignment inspectable.
    loc['location_id'] = city+':' + \
        loc.px_key.astype(str)+':'+loc.py_key.astype(str)
    row_labels = df[KEY+['year', 'weighted_mean_year', 'row_label_valid']].merge(
        loc[KEY+['location_id', 'label_eligible', 'age_class']], on=KEY, validate='many_to_one')
    in_period = df.year.between(MIN_OBSERVATION_YEAR, MAX_OBSERVATION_YEAR)
    coverage_ok = df.coverage.between(MIN_COVERAGE, 1)
    values, selected, valid_slots, incomplete_qa = select_coherent_observation(
        df)
    row_labels['in_observation_window'] = in_period.to_numpy()
    row_labels['coverage_eligible'] = coverage_ok.to_numpy()
    row_labels['selected_slot'] = selected
    keep = in_period.to_numpy() & coverage_ok.to_numpy() & (selected > 0)
    # No targets, label flags or building records enter the spectral table.
    spectra = df.loc[keep, KEY+['year']].copy()
    spectra[BANDS] = values[keep]
    spectra['n_valid_slots'] = valid_slots[keep]
    eligible_years = df.loc[in_period & coverage_ok].groupby(
        KEY).year.nunique().rename('n_eligible_years')
    audit = {'city': city, 'raw_rows': len(df), 'raw_locations': len(loc),
             'outside_observation_window_rows': int((~in_period).sum()),
             'invalid_coverage_rows': int((~coverage_ok).sum()),
             'no_complete_valid_slot_rows': int((selected == 0).sum()),
             'qa_valid_but_incomplete_slots': incomplete_qa, 'retained_spectral_rows': int(keep.sum()),
             'ineligible_label_locations': int((~loc.label_eligible).sum())}
    if EXPORT:
        # Preserve excluded rows with reason flags so label/filter decisions can be audited.
        row_labels.to_parquet(
            OUTPUT/f'{city.lower()}_row_age_labels.parquet', index=False)
        loc.to_parquet(
            OUTPUT/f'{city.lower()}_location_age_labels.parquet', index=False)
    return spectra, loc, eligible_years, audit

## 4. Deterministic per-location feature engineering

Aggregate only actual complete observations; no interpolation, backfill, or population fit is performed. Summaries describe each location's history through the configured cutoff:

- Overall means, standard deviations, medians, minima and maxima summarize central tendency and variability.
- Early (≤2003) and late (≥2004) means and their difference describe fixed-period change. A missing period stays missing.
- Least-squares slopes use actual acquisition years, so gaps do not masquerade as one-year steps. Slopes are descriptive associations, not detected construction dates.
- Counts and missingness flags describe observation availability. They form an optional feature group because they can encode city/sensor shortcuts.

Indices are disabled until upstream scaling and offset are confirmed. If enabled, they use corrected reflectance and are computed per scene before temporal aggregation. No feature depends on the target, class, or city name. Aggregation is safe before splitting because each location is processed independently with fixed rules.


In [ ]:
def build_features(spectra, eligible_years):
    s = spectra.copy()
    feature_signals = list(BANDS)
    if ENABLE_INDICES:
        reflectance = s[BANDS]*REFLECTANCE_SCALE+REFLECTANCE_OFFSET
        b, g, r, n, sw1, sw2 = [reflectance[x] for x in BANDS]
        def ratio(a, b): return a.div(b.where(b.abs() > 1e-6))
        s['NDVI'] = ratio(n-r, n+r)
        s['NDBI'] = ratio(sw1-n, sw1+n)
        s['MNDWI'] = ratio(g-sw1, g+sw1)
        s['BSI'] = ratio((sw1+r)-(n+b), (sw1+r)+(n+b))
        feature_signals += ['NDVI', 'NDBI', 'MNDWI', 'BSI']
    grouped = s.groupby(KEY, sort=True, observed=True)
    overall = grouped[feature_signals].agg(
        ['mean', 'std', 'median', 'min', 'max'])
    overall.columns = [f'{signal}_{stat}' for signal, stat in overall.columns]
    early = s.loc[s.year <= EARLY_END].groupby(
        KEY)[feature_signals].mean().add_suffix('_early_mean')
    late = s.loc[s.year > EARLY_END].groupby(
        KEY)[feature_signals].mean().add_suffix('_late_mean')
    features = overall.join(early).join(late)
    for signal in feature_signals:
        features[signal+'_late_minus_early'] = features[signal +
                                                        '_late_mean']-features[signal+'_early_mean']
    # Center years for numerical stability; actual gaps remain in the time coordinate.
    s['_t'] = s.year-(MIN_OBSERVATION_YEAR+MAX_OBSERVATION_YEAR)/2
    s['_t2'] = s._t**2
    time = s.groupby(KEY).agg(n=('_t', 'size'), sum_t=(
        '_t', 'sum'), sum_t2=('_t2', 'sum'))
    for signal in feature_signals:
        # Optional ratios can be missing: calculate the slope on finite pairs only.
        pair = s.loc[np.isfinite(s[signal]), KEY+['_t', '_t2', signal]].copy()
        pair['_ty'] = pair._t*pair[signal]
        stats = pair.groupby(KEY).agg(n=('_t', 'size'), t=('_t', 'sum'), t2=('_t2', 'sum'),
                                      y=(signal, 'sum'), ty=('_ty', 'sum'))
        denominator = stats.t2-stats.t**2/stats.n
        features[signal+'_slope_per_year'] = (stats.ty-stats.t*stats.y/stats.n).div(
            denominator.where((stats.n >= 2) & (denominator > 1e-10)))
    counts = time[['n']].rename(
        columns={'n': 'n_observed_years'}).join(eligible_years)
    early_count = s.loc[s.year <= EARLY_END].groupby(KEY).year.nunique()
    late_count = s.loc[s.year > EARLY_END].groupby(KEY).year.nunique()
    counts['n_early_years'] = early_count.reindex(counts.index, fill_value=0)
    counts['n_late_years'] = late_count.reindex(counts.index, fill_value=0)
    counts['fraction_expected_years_observed'] = counts.n_observed_years / \
        (MAX_OBSERVATION_YEAR-MIN_OBSERVATION_YEAR+1)
    counts['no_early_data'] = counts.n_early_years.eq(0).astype(int)
    counts['no_late_data'] = counts.n_late_years.eq(0).astype(int)
    counts['mean_valid_slots'] = grouped.n_valid_slots.mean()
    return features.join(counts).replace([np.inf, -np.inf], np.nan).reset_index(), feature_signals

## 5. Run preprocessing and inspect data retention

Processing cities sequentially limits peak memory. Attach targets only after feature construction, then retain one row per location. Locations with no usable spectra are absent from the feature table but remain in the exported label tables. Locations with unusable labels are retained in the full feature table and excluded from supervised subsets.

Audit counters are independent reason counts and can overlap; they should not be added as if they were a sequential removal funnel.


In [ ]:
tables, supervised, audit_rows = {}, {}, []
for city, path in PATHS.items():
    if not path.is_file():
        raise FileNotFoundError(path)
    spectra, labels, eligible_years, city_audit = prepare_city(city, path)
    features, signals = build_features(spectra, eligible_years)
    merged = features.merge(labels, on=KEY, how='left', validate='one_to_one')
    assert not merged.location_id.duplicated().any()
    tables[city] = merged.sort_values(KEY).reset_index(drop=True)
    supervised[city] = tables[city].loc[lambda x:x.age_class.notna()
                                        ].copy().reset_index(drop=True)
    supervised[city]['age_class'] = supervised[city].age_class.astype(int)
    city_audit['feature_locations'] = len(merged)
    city_audit['supervised_locations'] = len(supervised[city])
    audit_rows.append(city_audit)
    del spectra, labels, features, merged
AUDIT = pd.DataFrame(audit_rows).set_index('city')
display(AUDIT.T)
CLASS_COUNTS = pd.DataFrame({c: t.age_class.value_counts().reindex([1, 2, 3, 4], fill_value=0)
                             for c, t in supervised.items()}).astype(int)
display(CLASS_COUNTS.rename_axis('age_class'))
CLASS_COUNTS.plot.bar(
    rot=0, title='Supervised locations after preprocessing', figsize=(10, 4))
plt.ylabel('Unique locations')
plt.tight_layout()
plt.show()

## 6. Feature groups and metadata separation

The default predictor set is the spectral and temporal feature group. Availability indicators remain exported for controlled ablations but are not included by default. City, coordinates, IDs, construction years and target-validity flags are metadata only. Preserve missing features until the training-only imputation step; missing entire periods carry genuine uncertainty.

These features describe the full allowed retrospective history. They must not be reused unchanged for an earlier prediction date: rerun feature construction with that cutoff.


In [ ]:
SPECTRAL_FEATURES = [f'{signal}_{suffix}' for signal in signals for suffix in
                     ['mean', 'std', 'median', 'min', 'max', 'early_mean', 'late_mean', 'late_minus_early', 'slope_per_year']]
AVAILABILITY_FEATURES = ['n_observed_years', 'n_eligible_years', 'n_early_years', 'n_late_years',
                         'fraction_expected_years_observed', 'no_early_data', 'no_late_data', 'mean_valid_slots']
FEATURE_COLS = SPECTRAL_FEATURES
FORBIDDEN = {'age_class', 'weighted_mean_year', 'label_eligible', 'all_rows_label_valid',
             'target_distinct_including_null', 'city', 'px_key', 'py_key', 'location_id'}
assert not FORBIDDEN.intersection(FEATURE_COLS)
assert len(FEATURE_COLS) == len(set(FEATURE_COLS))
for city, table in supervised.items():
    assert set(FEATURE_COLS).issubset(table.columns)
    assert table[FEATURE_COLS].select_dtypes(
        include='number').shape[1] == len(FEATURE_COLS)
    assert not np.isinf(table[FEATURE_COLS].to_numpy()).any()
    assert table.n_observed_years.between(
        1, MAX_OBSERVATION_YEAR-MIN_OBSERVATION_YEAR+1).all()
    print(city, 'predictor shape:', table[FEATURE_COLS].shape)
    display(table[FEATURE_COLS].isna().mean().sort_values(
        ascending=False).head(12).to_frame('missing_fraction'))
print('Default predictor columns:', len(FEATURE_COLS))

## 7. Spatial folds for Madrid evaluation and Amsterdam adaptation

Assign fixed grid blocks in coordinate-key space to five folds using `StratifiedGroupKFold`. Every location in a block stays in one fold, and class proportions are approximately balanced. Coordinates define splits only; they are not predictors. The block size is a configurable starting point, not a validated physical independence distance.

For each outer fold, use `fold != k` for training/adaptation and `fold == k` for validation. Neighboring blocks can touch, so residual spatial autocorrelation remains; stronger validation may need buffers and building-footprint groups. These are development folds, not organizer-held-out test sets. Hyperparameter tuning still needs separate inner folds within each outer training pool.


In [ ]:
FOLD_COUNTS = {}
for city, table in supervised.items():
    bx = np.floor_divide(table.px_key.to_numpy(), BLOCK_SIZE_KEYS)
    by = np.floor_divide(table.py_key.to_numpy(), BLOCK_SIZE_KEYS)
    table['spatial_group'] = [f'{city}:{x}:{y}' for x, y in zip(bx, by)]
    table['fold'] = -1
    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    if table.spatial_group.nunique() < N_FOLDS:
        raise ValueError('Not enough geographic blocks')
    for fold, (train_idx, val_idx) in enumerate(splitter.split(table[FEATURE_COLS], table.age_class, table.spatial_group)):
        table.loc[val_idx, 'fold'] = fold
        assert set(table.iloc[train_idx].spatial_group).isdisjoint(
            table.iloc[val_idx].spatial_group)
    assert table.fold.ge(0).all()
    assert table.groupby('spatial_group').fold.nunique().max() == 1
    counts = pd.crosstab(table.fold, table.age_class).reindex(
        columns=[1, 2, 3, 4], fill_value=0)
    assert counts.gt(0).all().all(
    ), f'{city}: validation fold lacks a class; revise spatial design'
    FOLD_COUNTS[city] = counts
    print(city, 'validation locations per class / fold:')
    display(counts)

## 8. Reproducible Amsterdam few-shot subsets

For every validation fold, sample exactly 5, 25, 50, 100 and 200 **unique Amsterdam locations per class** from the remaining folds. Budgets are nested within a fold/repetition: the same randomized per-class ordering is reused, so the 5-example set is contained in the 25-example set. This reduces sampling noise in comparisons. An infeasible budget raises an error rather than duplicating pixels.

The exported membership table holds IDs, not fitted features. No evaluation location or spatial block is allowed into its adaptation set. Different budgets/folds intentionally reuse locations, but every particular evaluation remains disjoint. This notebook provides one seeded repetition; repeat seeds during modelling to assess few-shot sampling variability.


In [ ]:
def few_shot_membership(table, budgets=BUDGETS, seed=SEED, repetition=0):
    rows = []
    for fold in range(N_FOLDS):
        pool = table.loc[table.fold.ne(fold)]
        evaluation = table.loc[table.fold.eq(fold)]
        rng = np.random.default_rng(
            np.random.SeedSequence([seed, repetition, fold]))
        orders = {}
        for cls in [1, 2, 3, 4]:
            candidates = pool.loc[pool.age_class.eq(
                cls), 'location_id'].sort_values().to_numpy()
            if len(candidates) < max(budgets):
                raise ValueError(
                    f'Fold {fold}, class {cls}: insufficient examples')
            orders[cls] = rng.permutation(candidates)
        for n in budgets:
            ids = np.concatenate([orders[cls][:n] for cls in [1, 2, 3, 4]])
            sample = pool.set_index('location_id').loc[ids].reset_index()
            assert sample.age_class.value_counts().reindex(
                [1, 2, 3, 4]).eq(n).all()
            assert set(sample.location_id).isdisjoint(evaluation.location_id)
            assert set(sample.spatial_group).isdisjoint(
                evaluation.spatial_group)
            part = sample[['location_id', 'age_class']].copy()
            part['validation_fold'] = fold
            part['samples_per_class'] = n
            part['repetition'] = repetition
            rows.append(part)
    return pd.concat(rows, ignore_index=True)


FEW_SHOT = few_shot_membership(supervised['Amsterdam'])
display(FEW_SHOT.groupby(
    ['validation_fold', 'samples_per_class', 'age_class']).size().unstack('age_class'))
assert FEW_SHOT.equals(few_shot_membership(supervised['Amsterdam']))
for fold in range(N_FOLDS):
    previous = set()
    for n in sorted(BUDGETS):
        current = set(FEW_SHOT.loc[FEW_SHOT.validation_fold.eq(
            fold) & FEW_SHOT.samples_per_class.eq(n), 'location_id'])
        assert previous.issubset(current)
        previous = current

## 9. Training-only imputation and scaling

A median imputer and standard scaler are packaged together so the same fitted operations are applied to validation and target features. `keep_empty_features=True` preserves the column schema even if a training fold has an entirely missing period feature; such a column is filled with zero. No-validation-statistics means that validation means need not be zero.

Here we fit **only Madrid folds 1–4** and transform Madrid fold 0 plus all Amsterdam locations using those fixed source parameters. These transformed arrays are a demonstration tied to this exact split, not final cross-validation input. During model evaluation, place a **fresh** preprocessor in the full model pipeline for each outer/inner fold. Random forests do not require scaling; set `scale=False` for them. A transferred representation should normally retain its source scaler; do not refit it on Amsterdam evaluation data or silently change the source feature coordinate system.

Class weights are computed from the Madrid training subset only. Avoid global oversampling or balancing before splitting; the few-shot subsets are already balanced by construction.


In [ ]:
def make_preprocessor(scale=True):
    return Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
                     ('scaler', StandardScaler() if scale else 'passthrough')])


UNFITTED_PREPROCESSOR = make_preprocessor()
source = supervised['Madrid']
TRAIN_MASK = source.fold.ne(0)
VALID_MASK = source.fold.eq(0)
X_train_raw = source.loc[TRAIN_MASK, FEATURE_COLS]
X_validation_raw = source.loc[VALID_MASK, FEATURE_COLS]
SOURCE_FOLD0_PREPROCESSOR = clone(UNFITTED_PREPROCESSOR)
X_train = SOURCE_FOLD0_PREPROCESSOR.fit_transform(X_train_raw)
X_validation = SOURCE_FOLD0_PREPROCESSOR.transform(X_validation_raw)
X_amsterdam_source_scaled = SOURCE_FOLD0_PREPROCESSOR.transform(
    supervised['Amsterdam'][FEATURE_COLS])
y_train = source.loc[TRAIN_MASK, 'age_class'].to_numpy()
y_validation = source.loc[VALID_MASK, 'age_class'].to_numpy()
train_sample_weights = compute_sample_weight(
    class_weight='balanced', y=y_train)
for name, array in [('Madrid training', X_train), ('Madrid fold 0 validation', X_validation),
                    ('Amsterdam transformed with source training statistics', X_amsterdam_source_scaled)]:
    assert array.shape[1] == len(FEATURE_COLS) and np.isfinite(array).all()
    print(name, array.shape)
# Fresh numerical verification that the scaler learned only the training slice.
imputed_train = SOURCE_FOLD0_PREPROCESSOR.named_steps['imputer'].transform(
    X_train_raw)
np.testing.assert_allclose(
    SOURCE_FOLD0_PREPROCESSOR.named_steps['scaler'].mean_, imputed_train.mean(axis=0))
assert np.allclose(X_train.mean(axis=0), 0, atol=1e-8)
print('Training-only imputation/scaling verified; no classifier has been fitted.')

## 10. Verification of scientific edge cases

Small synthetic checks exercise meaningful failure modes independent of the current files: target boundaries, no mixed-scene band vectors, missing early periods, irregular-year slopes, and feature independence from label metadata. They supplement full-data integrity checks and are intentionally embedded in the notebook, so a clean run verifies assumptions after any edit.

The synthetic slope example increases Blue by 2 units per year over an irregular time gap. If a one-row step were mistaken for one year, it would incorrectly return 20 instead of 2.


In [ ]:
# Build features without supplying a target: it cannot access target construction dates.
toy = pd.DataFrame({'city': ['Toy', 'Toy'], 'px_key': [1, 1], 'py_key': [2, 2],
                    'year': [2005, 2015], 'n_valid_slots': [1, 1]})
for band in BANDS:
    toy[band] = [10., 30.]
toy_eligible = toy.groupby(KEY).year.nunique().rename('n_eligible_years')
toy_features, _ = build_features(toy, toy_eligible)
assert toy_features.Blue_early_mean.isna().all()
assert toy_features.Blue_late_minus_early.isna().all()
assert toy_features.no_early_data.eq(1).all()
assert np.isclose(toy_features.Blue_slope_per_year.iloc[0], 2.)
one, _ = build_features(toy.iloc[:1], toy_eligible)
assert one.Blue_std.isna().all() and one.Blue_slope_per_year.isna().all()
# Exercise the production selector: an incomplete earlier scene must be skipped.
corrupt = pd.DataFrame(
    {f'{band}_{slot}': [float(slot)] for band in BANDS for slot in [1, 2, 3]})
for slot in [1, 2, 3]:
    corrupt[f'qa_valid_{slot}'] = True
corrupt['Green_1'] = np.nan
values, chosen, counts, incomplete = select_coherent_observation(corrupt)
assert chosen.tolist() == [2] and np.all(values == 2) and incomplete == 1
assert counts.tolist() == [2]
for slot in [1, 2, 3]:
    corrupt[f'qa_valid_{slot}'] = False
values, chosen, counts, incomplete = select_coherent_observation(corrupt)
assert chosen.tolist() == [0] and np.isnan(
    values).all() and counts.tolist() == [0]
assert all(not t.location_id.duplicated().any() for t in supervised.values())
print('Boundary, missing-period, irregular-year slope, coherent-slot and full-data integrity checks passed.')

## 11. Export versioned artifacts and verify round-trip integrity

Save unscaled feature tables as the primary modelling input, along with labels, folds, sample membership, a feature schema and exact configuration. The unfitted template is suitable for cloning inside a model pipeline; the separate fitted artifact is explicitly named for Madrid fold 0. Loading joblib/pickle artifacts should be restricted to trusted files and matching package versions.

Input SHA-256 hashes and package versions make the run traceable to exact source files. Exports overwrite only files in this notebook's dedicated output folder on rerun. A JSON manifest records configuration and output hashes. Parquet round-trip checks confirm target/ID alignment, fold persistence and numerical features rather than merely checking file existence.


In [ ]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(
            lambda: stream.read(8*1024*1024), b''): digest.update(chunk)
    return digest.hexdigest()


if EXPORT:
    for city, table in tables.items():
        table.to_parquet(
            OUTPUT/f'{city.lower()}_all_location_features.parquet', index=False)
    for city, table in supervised.items():
        table.to_parquet(
            OUTPUT/f'{city.lower()}_supervised_features.parquet', index=False)
        table[KEY+['location_id', 'age_class', 'spatial_group', 'fold']
              ].to_csv(OUTPUT/f'{city.lower()}_spatial_folds.csv', index=False)
    FEW_SHOT.to_csv(OUTPUT/'amsterdam_few_shot_membership.csv', index=False)
    AUDIT.to_csv(OUTPUT/'preprocessing_audit.csv')
    CLASS_COUNTS.to_csv(OUTPUT/'age_class_counts.csv')
    joblib.dump(UNFITTED_PREPROCESSOR, OUTPUT/'unfitted_preprocessor.joblib')
    joblib.dump(SOURCE_FOLD0_PREPROCESSOR, OUTPUT /
                'madrid_fold0_training_preprocessor.joblib')
    np.savez_compressed(OUTPUT/'madrid_fold0_demo_arrays.npz', X_train=X_train, X_validation=X_validation,
                        y_train=y_train, y_validation=y_validation, train_sample_weights=train_sample_weights,
                        train_location_ids=source.loc[TRAIN_MASK, 'location_id'].to_numpy(
                            dtype=str),
                        validation_location_ids=source.loc[VALID_MASK, 'location_id'].to_numpy(dtype=str))
    # Persist metadata at the same time as arrays, avoiding reliance on notebook state.
    configuration = {'seed': SEED, 'class_events': CLASS_EVENTS, 'construction_year_range': [MIN_CONSTRUCTION_YEAR, MAX_CONSTRUCTION_YEAR],
                     'label_bins_right_closed': True, 'observation_year_range': [MIN_OBSERVATION_YEAR, MAX_OBSERVATION_YEAR],
                     'early_end': EARLY_END, 'min_coverage': MIN_COVERAGE, 'n_folds': N_FOLDS, 'block_size_key_units': BLOCK_SIZE_KEYS,
                     'budgets': BUDGETS, 'indices_enabled': ENABLE_INDICES, 'reflectance_scale': REFLECTANCE_SCALE,
                     'reflectance_offset': REFLECTANCE_OFFSET, 'spectral_feature_names': SPECTRAL_FEATURES,
                     'availability_feature_names': AVAILABILITY_FEATURES, 'default_feature_names': FEATURE_COLS,
                     'default_features_scaled': False, 'demo_training_city': 'Madrid', 'demo_validation_fold': 0,
                     'versions': VERSIONS, 'inputs': {c: {'path': str(p.relative_to(ROOT)), 'sha256': sha256(p)} for c, p in PATHS.items()}}
    (OUTPUT/'configuration.json').write_text(json.dumps(configuration, indent=2)+'\n')
    for city, table in supervised.items():
        restored = pd.read_parquet(
            OUTPUT/f'{city.lower()}_supervised_features.parquet')
        pd.testing.assert_frame_equal(table, restored)
    restored_preprocessor = joblib.load(
        OUTPUT/'madrid_fold0_training_preprocessor.joblib')
    np.testing.assert_allclose(
        restored_preprocessor.transform(X_validation_raw), X_validation)
    restored_membership = pd.read_csv(
        OUTPUT/'amsterdam_few_shot_membership.csv')
    pd.testing.assert_frame_equal(FEW_SHOT, restored_membership)
    artifacts = [p for p in OUTPUT.iterdir() if p.is_file()
                 and p.name != 'manifest.json']
    manifest = {p.name: {'bytes': p.stat().st_size, 'sha256': sha256(p)}
                for p in sorted(artifacts)}
    (OUTPUT/'manifest.json').write_text(json.dumps(manifest, indent=2)+'\n')
    print('Export and round-trip verification passed:', OUTPUT)
    display(pd.DataFrame(manifest).T[['bytes']])
else:
    print('Export disabled; results are in memory only.')

## 12. How to use the prepared data for modelling

Read `configuration.json` for the ordered `default_feature_names`, then read a city's `*_supervised_features.parquet`. `X = table[feature_names]`; `y = table['age_class']`. For outer fold `k`, training/adaptation comes from `table.fold != k` and validation from `table.fold == k`. Keep `spatial_group` for inner validation. Clone the unfitted preprocessor and place it ahead of the estimator in a fresh full `Pipeline` for each training run.

For Amsterdam few-shot experiments, select membership by `validation_fold`, `samples_per_class` and `repetition`, then join on `location_id`. Membership contains exactly `n` labels per class. Transfer the Madrid source model using those labels; **do not fit on the remaining Amsterdam evaluation labels**. This notebook prepares the inputs; it does not implement or evaluate a transfer algorithm.

**Remaining decisions and limitations:** confirm city events and boundary inclusivity; confirm reflectance encoding before adding indices; assess grid block size/buffers; compare spectral-only versus availability features using training folds; report macro F1 and class-level metrics across five outer folds during modelling. No performance claim follows from preprocessing alone. All five budgets are retained despite inconsistent presentation prose in the introduction.